## Load data

In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import pandas as pd

EXP_TAGS_DEFAULT = ("exp1", "exp3", "exp4", "exp5")
PROFILE_DEFAULT = "small"  # run_experiments.sh all small 기준

SMALL_PROFILE_MODELS = {"GCN", "GAT"}
SMALL_PROFILE_LAYERS = {2, 4}
SMALL_PROFILE_DATASETS = {
    "cora_public",
    "citeseer_public",
    "texas",
    "cornell",
}

EXP_SETTING_KEYS = {
    "exp1": [
        "dataset",
        "model",
        "num_layers",
        "ratio_group_elem",
        "num_group_elem",
        "seed",
    ],
    "exp3": [
        "dataset",
        "model",
        "num_layers",
        "removal_candidate_sampler",
        "removal_neighbor_dist",
        "num_group_elem",
        "seed",
    ],
    "exp4": [
        "dataset",
        "model",
        "num_layers",
        "num_of_clusters",
        "edges_per_cluster",
        "cluster_ratio_percent",
        "intra_cluster_dist",
        "inter_cluster_dist",
        "influence_mode",
        "num_group_elem",
        "seed",
    ],
    "exp5": [
        "dataset",
        "model",
        "num_layers",
        "num_of_clusters",
        "edges_per_cluster",
        "cluster_ratio_percent",
        "intra_cluster_dist",
        "inter_cluster_dist",
        "influence_mode",
        "num_group_elem",
        "seed",
    ],
}

CLUSTER_SIGNATURE_KEYS = [
    "dataset",
    "model",
    "num_layers",
    "num_of_clusters",
    "edges_per_cluster",
    "cluster_ratio_percent",
    "intra_cluster_dist",
    "inter_cluster_dist",
    "num_group_elem",
    "seed",
]

EXP5_CLUSTER_RATIO_PERCENTS = {9, 15}


def _as_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _get_influence_mode(cfg: Dict) -> str | None:
    return cfg.get("influence_mode") or cfg.get("influence_calculation_mode")


def _is_exp5_cluster_setting(cfg: Dict) -> bool:
    return (
        cfg.get("experiment_name") == "clusters"
        and _as_int(cfg.get("num_of_clusters")) == 3
        and _as_int(cfg.get("intra_cluster_dist")) == 1
        and _as_int(cfg.get("inter_cluster_dist")) == 2
        and _as_int(cfg.get("cluster_ratio_percent")) in EXP5_CLUSTER_RATIO_PERCENTS
    )


def _cluster_signature(cfg: Dict) -> Tuple:
    return tuple(cfg.get(key) for key in CLUSTER_SIGNATURE_KEYS)


def _pick_calc_column(df: pd.DataFrame, cfg: Dict) -> str | None:
    mode = _get_influence_mode(cfg)
    if mode == "clusterwise_step_by_step":
        if "clusterwise_step_by_step_mean_total" in df.columns:
            return "clusterwise_step_by_step_mean_total"
        if "clusterwise_fixed_theta_total" in df.columns:
            return "clusterwise_fixed_theta_total"

    for col in [
        "calculate_influence_total",
        "clusterwise_fixed_theta_total",
        "clusterwise_step_by_step_mean_total",
    ]:
        if col in df.columns:
            return col
    return None


def _pick_actual_column(df: pd.DataFrame) -> Tuple[str | None, str | None]:
    if "pbrf_total" in df.columns:
        return "pbrf_total", "pbrf_total"
    if "leave_k_out" in df.columns:
        return "leave_k_out", "leave_k_out"
    return None, None


def _classify_exp_tag(cfg: Dict, exp5_signatures: set[Tuple]) -> str | None:
    exp_name = cfg.get("experiment_name")
    if exp_name == "large_drop_influence":
        return "exp1"
    if exp_name == "non_neighbor_edges":
        return "exp3"
    if exp_name == "clusters":
        # exp4/exp5 모두 experiment_name=clusters라 config rule로 분리한다.
        # exp5의 calculate_influence run도 가져오기 위해,
        # 같은 세팅에서 step_by_step run이 존재하면 exp5로 묶는다.
        if _is_exp5_cluster_setting(cfg) and _cluster_signature(cfg) in exp5_signatures:
            return "exp5"
        return "exp4"
    return None


def _matches_profile(cfg: Dict, profile: str) -> bool:
    if profile == "full":
        return True
    if profile != "small":
        raise ValueError("profile must be one of {'full', 'small'}")

    model = str(cfg.get("model", "")).upper()
    layer = cfg.get("num_layers")
    dataset = str(cfg.get("dataset", "")).lower()

    return (
        model in SMALL_PROFILE_MODELS
        and layer in SMALL_PROFILE_LAYERS
        and dataset in SMALL_PROFILE_DATASETS
    )


def load_experiment_candidates(
    results_root: str | Path = "results",
    exp_tags: Iterable[str] = EXP_TAGS_DEFAULT,
    profile: str = PROFILE_DEFAULT,
) -> pd.DataFrame:
    results_root = Path(results_root)
    requested_tags = set(exp_tags)

    raw_run_entries: List[Dict] = []
    for cfg_path in sorted(results_root.rglob("config.json")):
        try:
            cfg = json.loads(cfg_path.read_text())
        except Exception:
            continue

        if not _matches_profile(cfg, profile):
            continue

        result_dir = cfg_path.parent
        candidate_csvs = [
            path
            for path in [
                result_dir / "candidate_results.csv",
                result_dir / "removal_candidate_results.csv",
                result_dir / "insertion_candidate_results.csv",
            ]
            if path.exists()
        ]
        if not candidate_csvs:
            continue

        raw_run_entries.append(
            {
                "result_dir": str(result_dir),
                "config": cfg,
                "candidate_csvs": candidate_csvs,
            }
        )

    exp5_signatures = {
        _cluster_signature(entry["config"])
        for entry in raw_run_entries
        if _is_exp5_cluster_setting(entry["config"])
        and _get_influence_mode(entry["config"]) == "clusterwise_step_by_step"
    }

    run_entries: List[Dict] = []
    for entry in raw_run_entries:
        cfg = entry["config"]
        exp_tag = _classify_exp_tag(cfg, exp5_signatures)
        if exp_tag is None or exp_tag not in requested_tags:
            continue

        setting_keys = EXP_SETTING_KEYS[exp_tag]
        setting_payload = {key: cfg.get(key) for key in setting_keys}
        if setting_payload.get("influence_mode") is None:
            setting_payload["influence_mode"] = _get_influence_mode(cfg)
        setting_key = json.dumps(setting_payload, sort_keys=True, ensure_ascii=True)

        run_entries.append(
            {
                "exp_tag": exp_tag,
                "exp_name": cfg.get("experiment_name"),
                "result_dir": entry["result_dir"],
                "setting_key": setting_key,
                "config": cfg,
                "candidate_csvs": entry["candidate_csvs"],
            }
        )

    run_entries.sort(
        key=lambda entry: (
            entry["exp_tag"],
            entry["setting_key"],
            entry["result_dir"],
        )
    )

    run_repeat_counter = defaultdict(int)
    rows: List[Dict] = []

    for run in run_entries:
        counter_key = (run["exp_tag"], run["setting_key"])
        run_repeat_idx = run_repeat_counter[counter_key]
        run_repeat_counter[counter_key] += 1

        cfg = run["config"]
        mode = _get_influence_mode(cfg)

        for csv_path in run["candidate_csvs"]:
            try:
                cand_df = pd.read_csv(csv_path)
            except Exception:
                continue
            if cand_df.empty:
                continue

            calc_col = _pick_calc_column(cand_df, cfg)
            actual_col, actual_source = _pick_actual_column(cand_df)

            for _, row in cand_df.iterrows():
                rows.append(
                    {
                        "exp": run["exp_tag"],
                        "experiment_name": run["exp_name"],
                        "result_dir": run["result_dir"],
                        "candidate_table": csv_path.name,
                        "run_repeat_idx": run_repeat_idx,
                        "setting_key": run["setting_key"],
                        "candidate_idx": row.get("candidate_idx"),
                        "candidate_edges": row.get("candidate_edges"),
                        "num_edges": row.get("num_edges"),
                        "calced_inf": row.get(calc_col) if calc_col else pd.NA,
                        "actual_inf": row.get(actual_col) if actual_col else pd.NA,
                        "actual_source": actual_source,
                        "config": cfg,
                        "cfg_dataset": cfg.get("dataset"),
                        "cfg_model": cfg.get("model"),
                        "cfg_num_layers": cfg.get("num_layers"),
                        "cfg_ratio_group_elem": cfg.get("ratio_group_elem"),
                        "cfg_removal_candidate_sampler": cfg.get("removal_candidate_sampler"),
                        "cfg_removal_neighbor_dist": cfg.get("removal_neighbor_dist"),
                        "cfg_num_of_clusters": cfg.get("num_of_clusters"),
                        "cfg_edges_per_cluster": cfg.get("edges_per_cluster"),
                        "cfg_cluster_ratio_percent": cfg.get("cluster_ratio_percent"),
                        "cfg_intra_cluster_dist": cfg.get("intra_cluster_dist"),
                        "cfg_inter_cluster_dist": cfg.get("inter_cluster_dist"),
                        "cfg_influence_mode": mode,
                        "cfg_num_group_elem": cfg.get("num_group_elem"),
                        "cfg_seed": cfg.get("seed"),
                    }
                )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    sort_cols = [
        "exp",
        "cfg_dataset",
        "cfg_model",
        "cfg_num_layers",
        "run_repeat_idx",
        "candidate_table",
        "candidate_idx",
    ]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    return df


df_all = load_experiment_candidates(
    "results",
    exp_tags=EXP_TAGS_DEFAULT,
    profile=PROFILE_DEFAULT,
)
df_exp1 = df_all[df_all["exp"] == "exp1"].reset_index(drop=True)
df_exp3 = df_all[df_all["exp"] == "exp3"].reset_index(drop=True)
df_exp4 = df_all[df_all["exp"] == "exp4"].reset_index(drop=True)
df_exp5 = df_all[df_all["exp"] == "exp5"].reset_index(drop=True)

print(f"profile: {PROFILE_DEFAULT}")
print(f"df_all shape: {df_all.shape}")
print(f"df_exp1 shape: {df_exp1.shape}")
print(f"df_exp3 shape: {df_exp3.shape}")
print(f"df_exp4 shape: {df_exp4.shape}")
print(f"df_exp5 shape: {df_exp5.shape}")

print(df_all.head(5))

repeat_summary = (
    df_all.groupby(["exp", "setting_key"])["result_dir"]
    .nunique()
    .reset_index(name="num_repeats")
    .groupby("exp")["num_repeats"]
    .describe()
)
print(repeat_summary)


profile: small
df_all shape: (12150, 27)
df_exp1 shape: (6850, 27)
df_exp3 shape: (2200, 27)
df_exp4 shape: (2150, 27)
df_exp5 shape: (950, 27)
    exp       experiment_name  \
0  exp1  large_drop_influence   
1  exp1  large_drop_influence   
2  exp1  large_drop_influence   
3  exp1  large_drop_influence   
4  exp1  large_drop_influence   

                                          result_dir        candidate_table  \
0  results/GNH/mean_validation_loss/GAT/linear_Fa...  candidate_results.csv   
1  results/GNH/mean_validation_loss/GAT/linear_Fa...  candidate_results.csv   
2  results/GNH/mean_validation_loss/GAT/linear_Fa...  candidate_results.csv   
3  results/GNH/mean_validation_loss/GAT/linear_Fa...  candidate_results.csv   
4  results/GNH/mean_validation_loss/GAT/linear_Fa...  candidate_results.csv   

   run_repeat_idx                                        setting_key  \
0               0  {"dataset": "citeseer_public", "influence_mode...   
1               0  {"dataset": "citese

## Exp 1

In [2]:
df1 = df_exp1.drop(columns=["candidate_table", "config", "setting_key", "experiment_name", "result_dir", "candidate_table", "setting_key", "candidate_idx", "candidate_edges"], inplace=False)
df1["inf_mae"] = (df1["calced_inf"] - df1["actual_inf"]).abs()
df1["inf_mape"] = (df1["calced_inf"] - df1["actual_inf"]).abs() / df1["actual_inf"].abs().replace(0, pd.NA) * 100

corr_group_keys = ["cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_ratio_group_elem", "run_repeat_idx"]
corr_by_run = (
    df1.groupby(corr_group_keys, dropna=False)
    .apply(lambda g: g["calced_inf"].corr(g["actual_inf"], method="spearman"))
    .rename("inf_corr")
    .reset_index()
)
df1 = df1.merge(corr_by_run, on=corr_group_keys, how="left")

/tmp/ipykernel_2738205/1164478893.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1.groupby(corr_group_keys, dropna=False)


In [3]:
df1['inf_corr']

0       0.365282
1       0.400240
2       0.613349
3       0.526915
4       0.650612
          ...   
6845    0.252533
6846    0.688451
6847    0.320912
6848    0.432989
6849    0.352989
Name: inf_corr, Length: 6850, dtype: float64

In [4]:
df1.columns

Index(['exp', 'run_repeat_idx', 'num_edges', 'calced_inf', 'actual_inf',
       'actual_source', 'cfg_dataset', 'cfg_model', 'cfg_num_layers',
       'cfg_ratio_group_elem', 'cfg_removal_candidate_sampler',
       'cfg_removal_neighbor_dist', 'cfg_num_of_clusters',
       'cfg_edges_per_cluster', 'cfg_cluster_ratio_percent',
       'cfg_intra_cluster_dist', 'cfg_inter_cluster_dist',
       'cfg_influence_mode', 'cfg_num_group_elem', 'cfg_seed', 'inf_mae',
       'inf_mape', 'inf_corr'],
      dtype='object')

In [5]:
df1_res = (
    df1.groupby(["cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_ratio_group_elem"])[["inf_mae", "inf_mape", "calced_inf", "actual_inf", "inf_corr"]]
    .agg(["mean", "std"])
)
# MultiIndex columns -> tuple index로 평탄화 (build_mean_std_table와 호환)
df1_res.columns = pd.Index(df1_res.columns.to_flat_index())

In [6]:
df1_res

(inf_mae, mean)  \
cfg_dataset     cfg_model cfg_num_layers cfg_ratio_group_elem                    
citeseer_public GAT       2              1.0                          0.002790   
                                         5.0                          0.013269   
                                         10.0                         0.028059   
                                         20.0                         0.060605   
                                         30.0                         0.096648   
...                                                                        ...   
texas           GCN       4              20.0                         0.063073   
                                         30.0                         0.100050   
                                         50.0                         0.156623   
                                         70.0                         0.249057   
                                         90.0                         0.120049   

                                                               (inf_mae, std)  \
cfg_dataset     cfg_model cfg_num_layers cfg_ratio_group_elem                   
citeseer_public GAT       2              1.0                         0.002268   
                                         5.0                         0.005215   
                                         10.0                        0.007349   
                                         20.0                        0.011411   
                                         30.0                        0.012994   
...                                                                       ...   
texas           GCN       4              20.0                        0.050426   
                                         30.0                        0.063518   
                                         50.0                        0.106078   
                                         70.0                        0.125255   
                                         90.0                        0.090633   

                                                               (inf_mape, mean)  \
cfg_dataset     cfg_model cfg_num_layers cfg_ratio_group_elem                     
citeseer_public GAT       2              1.0                          82.189927   
                                         5.0                          92.166196   
                                         10.0                         87.132993   
                                         20.0                         83.797751   
                                         30.0                         82.729877   
...                                                                         ...   
texas           GCN       4              20.0                        441.864478   
                                         30.0                        841.404325   
                                         50.0                        584.544760   
                                         70.0                        585.932908   
                                         90.0                        110.645361   

                                                               (inf_mape, std)  \
cfg_dataset     cfg_model cfg_num_layers cfg_ratio_group_elem                    
citeseer_public GAT       2              1.0                         44.080358   
                                         5.0                         40.292090   
                                         10.0                        20.905422   
                                         20.0                         9.274096   
                                         30.0                         6.380507   
...                                                                        ...   
texas           GCN       4              20.0                       919.573222   
                                         30.0                      2632.431783   
                                         5

In [7]:
import pandas as pd
import numpy as np

# -----------------------------------
# 설정: 데이터셋 그룹 정의
# -----------------------------------
HOMOPHILIC_DATASETS = ["cora_public", "citeseer_public"]
HETEROPHILIC_DATASETS = ["texas", "cornell"]

# 원하는 순서
MODEL_ORDER = ["GCN", "GAT"]
LAYER_ORDER = [2, 4]
RATIO_ORDER = [1.0, 5.0, 10.0, 20.0, 30.0, 50.0, 70.0, 90.0]  # 예시 8개
DATASET_ORDER = HOMOPHILIC_DATASETS + HETEROPHILIC_DATASETS


# -----------------------------------
# 유틸: mean ± std 문자열 포맷
# -----------------------------------
def format_mean_std(mean, std, mean_fmt=".4f", std_fmt=".4f"):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:{mean_fmt}}"
    return f"{mean:{mean_fmt}} ± {std:{std_fmt}}"


# -----------------------------------
# 핵심 함수:
# df1_res -> row=ratio, col=(group, dataset, model, layers)
# -----------------------------------
def build_mean_std_table(
    df_res,
    metric_name="inf_mae",
    ratio_order=None,
    dataset_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    model_order=None,
    layer_order=None,
    mean_fmt=".4f",
    std_fmt=".4f",
):
    # df_res: MultiIndex index + MultiIndex columns 형태 가정
    # 예: df_res[("inf_error", "mean")], df_res[("inf_error", "std")]

    df = df_res.copy().reset_index()

    # 컬럼 추출
    df["mean"] = df[(metric_name, "mean")]
    df["std"] = df[(metric_name, "std")]

    # group 컬럼 추가
    homophilic_datasets = homophilic_datasets or []
    heterophilic_datasets = heterophilic_datasets or []

    def dataset_group(ds):
        if ds in homophilic_datasets:
            return "Homophilic"
        elif ds in heterophilic_datasets:
            return "Heterophilic"
        return "Other"

    df["group"] = df["cfg_dataset"].map(dataset_group)

    # 문자열 셀 생성
    df["cell"] = df.apply(
        lambda r: format_mean_std(r["mean"], r["std"], std_fmt), axis=1
    )

    # pivot
    table = df.pivot(
        index="cfg_ratio_group_elem",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="cell",
    )

    # -----------------------------------
    # 순서 정리
    # -----------------------------------
    if ratio_order is not None:
        table = table.reindex(ratio_order)

    # MultiIndex column 정렬용 전체 조합 생성
    current_cols = list(table.columns)

    groups_in_order = []
    if homophilic_datasets:
        groups_in_order.append(("Homophilic", homophilic_datasets))
    if heterophilic_datasets:
        groups_in_order.append(("Heterophilic", heterophilic_datasets))

    desired_cols = []
    for group_name, ds_list in groups_in_order:
        ds_seq = dataset_order if dataset_order is not None else ds_list
        ds_seq = [ds for ds in ds_seq if ds in ds_list]
        for ds in ds_seq:
            for model in (model_order or sorted(df["cfg_model"].unique())):
                for layer in (layer_order or sorted(df["cfg_num_layers"].unique())):
                    col = (group_name, ds, model, layer)
                    if col in current_cols:
                        desired_cols.append(col)

    # 혹시 Other가 있으면 뒤에 추가
    remaining_cols = [c for c in current_cols if c not in desired_cols]
    desired_cols.extend(sorted(remaining_cols))

    table = table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

    # index 이름 보기 좋게
    table.index.name = "Drop Ratio (%)"
    table.columns.names = ["Graph Type", "Dataset", "Model", "#Layers"]

    return table


def build_mean_table(
    df_res,
    metric_name="inf_mae",
    ratio_order=None,
    dataset_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    model_order=None,
    layer_order=None,
    mean_fmt=".4f",
    std_fmt=".4f",
):
    df = df_res.copy().reset_index()

    # 컬럼 추출
    df["mean"] = df[(metric_name, "mean")]

    # group 컬럼 추가
    homophilic_datasets = homophilic_datasets or []
    heterophilic_datasets = heterophilic_datasets or []

    def dataset_group(ds):
        if ds in homophilic_datasets:
            return "Homophilic"
        elif ds in heterophilic_datasets:
            return "Heterophilic"
        return "Other"

    df["group"] = df["cfg_dataset"].map(dataset_group)

    # 문자열 셀 생성
    df["cell"] = df.apply(
        lambda r: format_mean_std(r["mean"], None, mean_fmt, None), axis=1
    )

    # pivot
    table = df.pivot(
        index="cfg_ratio_group_elem",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="cell",
    )

    # -----------------------------------
    # 순서 정리
    # -----------------------------------
    if ratio_order is not None:
        table = table.reindex(ratio_order)

    # MultiIndex column 정렬용 전체 조합 생성
    current_cols = list(table.columns)

    groups_in_order = []
    if homophilic_datasets:
        groups_in_order.append(("Homophilic", homophilic_datasets))
    if heterophilic_datasets:
        groups_in_order.append(("Heterophilic", heterophilic_datasets))

    desired_cols = []
    for group_name, ds_list in groups_in_order:
        ds_seq = dataset_order if dataset_order is not None else ds_list
        ds_seq = [ds for ds in ds_seq if ds in ds_list]
        for ds in ds_seq:
            for model in (model_order or sorted(df["cfg_model"].unique())):
                for layer in (layer_order or sorted(df["cfg_num_layers"].unique())):
                    col = (group_name, ds, model, layer)
                    if col in current_cols:
                        desired_cols.append(col)

    # 혹시 Other가 있으면 뒤에 추가
    remaining_cols = [c for c in current_cols if c not in desired_cols]
    desired_cols.extend(sorted(remaining_cols))

    table = table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

    # index 이름 보기 좋게
    table.index.name = "Drop Ratio (%)"
    table.columns.names = ["Graph Type", "Dataset", "Model", "#Layers"]

    return table

In [8]:
# -----------------------------
# 셀 포맷 함수
# -----------------------------
def pretty_num(x, fmt=None):
    if x is None or pd.isna(x):
        return ""
    x = float(x)

    if fmt is not None:
        return format(x, fmt)

    if x == 0:
        return "0"
    elif abs(x) < 0.01:
        return f"{x:.4f}"
    elif abs(x) < 1:
        return f"{x:.3f}"
    else:
        return f"{x:.2f}"

def latex_mean_std(mean, std, mean_fmt=None, std_fmt=None, bold=False):
    if pd.isna(mean):
        return "--"

    mean_str = pretty_num(mean, fmt=mean_fmt)
    std_str = pretty_num(std, fmt=std_fmt) if not pd.isna(std) else ""

    if std_str == "":
        return mean_str

    macro = r"\bmstd" if bold else r"\mstd"
    return f"{macro}{{{mean_str}}}{{{std_str}}}"


# -----------------------------
# group 구분
# -----------------------------
def dataset_group(ds, homophilic_datasets, heterophilic_datasets):
    if ds in homophilic_datasets:
        return "Homophilic"
    elif ds in heterophilic_datasets:
        return "Heterophilic"
    return "Other"


# -----------------------------
# 숫자 테이블 생성
# row: ratio
# col: (group, dataset, model, num_layers)
# -----------------------------
def build_numeric_tables(
    df_res,
    metric_name="inf_error",
    ratio_order=None,
    dataset_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    model_order=None,
    layer_order=None,
):
    df = df_res.copy().reset_index()

    df["mean"] = df[(metric_name, "mean")]
    df["std"] = df[(metric_name, "std")]

    homophilic_datasets = homophilic_datasets or []
    heterophilic_datasets = heterophilic_datasets or []

    df["group"] = df["cfg_dataset"].map(
        lambda ds: dataset_group(ds, homophilic_datasets, heterophilic_datasets)
    )

    mean_table = df.pivot(
        index="cfg_ratio_group_elem",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="mean",
    )

    std_table = df.pivot(
        index="cfg_ratio_group_elem",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="std",
    )

    if ratio_order is not None:
        mean_table = mean_table.reindex(ratio_order)
        std_table = std_table.reindex(ratio_order)

    current_cols = list(mean_table.columns)

    groups_in_order = []
    if homophilic_datasets:
        groups_in_order.append(("Homophilic", homophilic_datasets))
    if heterophilic_datasets:
        groups_in_order.append(("Heterophilic", heterophilic_datasets))

    desired_cols = []
    for group_name, ds_list in groups_in_order:
        ds_seq = dataset_order if dataset_order is not None else ds_list
        ds_seq = [ds for ds in ds_seq if ds in ds_list]
        for ds in ds_seq:
            for model in (model_order or sorted(df["cfg_model"].unique())):
                for layer in (layer_order or sorted(df["cfg_num_layers"].unique())):
                    col = (group_name, ds, model, layer)
                    if col in current_cols:
                        desired_cols.append(col)

    remaining_cols = [c for c in current_cols if c not in desired_cols]
    desired_cols.extend(sorted(remaining_cols))

    mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))
    std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

    mean_table.index.name = "Drop Ratio"
    std_table.index.name = "Drop Ratio"
    mean_table.columns.names = ["Graph Type", "Dataset", "Model", "#Layers"]
    std_table.columns.names = ["Graph Type", "Dataset", "Model", "#Layers"]

    return mean_table, std_table


def build_latex_cell_table(
    mean_table,
    std_table,
    mean_fmt=".4f",
    std_fmt=".4f",
    highlight=None,
):
    latex_table = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)

    if highlight not in [None, "min", "max"]:
        raise ValueError("highlight must be one of None, 'min', 'max'")

    bold_mask = pd.DataFrame(False, index=mean_table.index, columns=mean_table.columns)

    if highlight == "min":
        for col in mean_table.columns:
            col_values = mean_table[col].dropna()
            if len(col_values) > 0:
                min_val = col_values.min()
                bold_mask.loc[mean_table[col] == min_val, col] = True

    elif highlight == "max":
        for col in mean_table.columns:
            col_values = mean_table[col].dropna()
            if len(col_values) > 0:
                max_val = col_values.max()
                bold_mask.loc[mean_table[col] == max_val, col] = True

    for idx in mean_table.index:
        for col in mean_table.columns:
            mean = mean_table.loc[idx, col]
            std = std_table.loc[idx, col]
            bold = bold_mask.loc[idx, col]
            latex_table.loc[idx, col] = latex_mean_std(
                mean, std, mean_fmt=mean_fmt, std_fmt=std_fmt, bold=bold
            )

    latex_table.index.name = mean_table.index.name
    latex_table.columns.names = mean_table.columns.names
    return latex_table


# -----------------------------
# LaTeX table 문자열 생성
# -----------------------------
def dataframe_to_latex_table(
    latex_cell_table,
    caption="",
    label="",
    column_format=None,
    escape=False,
):
    n_data_cols = latex_cell_table.shape[1]

    if column_format is None:
        column_format = "l" + "c" * n_data_cols

    latex_str = latex_cell_table.to_latex(
        escape=escape,
        multicolumn=True,
        multirow=True,
        column_format=column_format,
        caption=caption if caption else None,
        label=label if label else None,
    )
    return latex_str


# -----------------------------
# 한번에 다 하는 wrapper
# -----------------------------
def make_latex_result_table(
    df_res,
    metric_name="inf_error",
    ratio_order=None,
    dataset_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    model_order=None,
    layer_order=None,
    mean_fmt=".4f",
    std_fmt=".4f",
    highlight=None,   # None / "min" / "max"
    caption="",
    label="",
    column_format=None,
):
    mean_table, std_table = build_numeric_tables(
        df_res=df_res,
        metric_name=metric_name,
        ratio_order=ratio_order,
        dataset_order=dataset_order,
        homophilic_datasets=homophilic_datasets,
        heterophilic_datasets=heterophilic_datasets,
        model_order=model_order,
        layer_order=layer_order,
    )

    latex_cell_table = build_latex_cell_table(
        mean_table=mean_table,
        std_table=std_table,
        mean_fmt=mean_fmt,
        std_fmt=std_fmt,
        highlight=highlight,
    )

    latex_str = dataframe_to_latex_table(
        latex_cell_table=latex_cell_table,
        caption=caption,
        label=label,
        column_format=column_format,
        escape=False,
    )

    return latex_cell_table, latex_str

In [9]:
latex_cell_table, latex_str = make_latex_result_table(
    df_res=df1_res,
    metric_name="inf_corr",
    ratio_order=RATIO_ORDER,
    dataset_order=DATASET_ORDER,
    homophilic_datasets=HOMOPHILIC_DATASETS,
    heterophilic_datasets=HETEROPHILIC_DATASETS,
    model_order=MODEL_ORDER,
    layer_order=LAYER_ORDER,
    mean_fmt=".2f",
    std_fmt=".4f",
    highlight=None,   # inf_error면 보통 작을수록 좋으니 min 추천
    caption="Influence error across drop ratios.",
    label="tab:inf_error_drop_ratio",
)

print(latex_str)

\begin{table}
\caption{Influence error across drop ratios.}
\label{tab:inf_error_drop_ratio}
\begin{tabular}{lcccccccccccccccc}
\toprule
Graph Type & \multicolumn{8}{r}{Homophilic} & \multicolumn{8}{r}{Heterophilic} \\
Dataset & \multicolumn{4}{r}{cora_public} & \multicolumn{4}{r}{citeseer_public} & \multicolumn{4}{r}{texas} & \multicolumn{4}{r}{cornell} \\
Model & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} \\
#Layers & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 \\
Drop Ratio &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1.000000 & \mstd{0.87}{0.0000} & \mstd{0.70}{0.0000} & \mstd{0.69}{0.0000} & \mstd{0.77}{0.0000} & \mstd{0.79}{0.0000} & \mstd{0.64}{0.0000} & \mstd{0.43}{0.0000} & \mstd{0.60}{0.0000} & \mstd{0.49}{0.0000} & \mstd{0.69}{0.0000} & \mstd{0.75}{0.0000} & \mstd{0.38}{0.0000} & \m

In [10]:
table_mae = build_mean_std_table(
    df_res=pd.DataFrame(df1_res),
    metric_name="inf_mae",
    ratio_order=RATIO_ORDER,
    dataset_order=DATASET_ORDER,
    homophilic_datasets=HOMOPHILIC_DATASETS,
    heterophilic_datasets=HETEROPHILIC_DATASETS,
    model_order=MODEL_ORDER,
    layer_order=LAYER_ORDER,
    mean_fmt=".4f",
    std_fmt=".4f",
)

table_mape = build_mean_std_table(
    df_res=pd.DataFrame(df1_res),
    metric_name="inf_mape",
    ratio_order=RATIO_ORDER,
    dataset_order=DATASET_ORDER,
    homophilic_datasets=HOMOPHILIC_DATASETS,
    heterophilic_datasets=HETEROPHILIC_DATASETS,
    model_order=MODEL_ORDER,
    layer_order=LAYER_ORDER,
    mean_fmt=".4f",
    std_fmt=".4f",
)

table_corr = build_mean_table(
    df_res=pd.DataFrame(df1_res),
    metric_name="inf_corr",
    ratio_order=RATIO_ORDER,
    dataset_order=DATASET_ORDER,
    homophilic_datasets=HOMOPHILIC_DATASETS,
    heterophilic_datasets=HETEROPHILIC_DATASETS,
    model_order=MODEL_ORDER,
    layer_order=LAYER_ORDER,
    mean_fmt=".4f",
    std_fmt=".4f",
)

In [11]:
styled_mae = (
    table_mae.style
    .set_caption("inf_mae (mean ± std)")
    .set_properties(**{
        "text-align": "center",
        "white-space": "nowrap",
        "font-size": "10pt"
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12pt"), ("font-weight", "bold")]},
    ])
)

styled_mape = (
    table_mape.style
    .set_caption("inf_mape (mean ± std)")
    .set_properties(**{
        "text-align": "center",
        "white-space": "nowrap",
        "font-size": "10pt"
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12pt"), ("font-weight", "bold")]},
    ])
)

styled_corr = (
    table_corr.style
    .set_caption("inf_corr")
    .set_properties(**{
        "text-align": "center",
        "white-space": "nowrap",
        "font-size": "10pt"
    })
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "12pt"), ("font-weight", "bold")]},
    ])
)

In [12]:
table_mae

Graph Type           Homophilic                                    \
Dataset             cora_public                                     
Model                       GCN                               GAT   
#Layers                       2                4                2   
Drop Ratio (%)                                                      
1.0             0.0004 ± 0.0005  0.0021 ± 0.0017  0.0013 ± 0.0016   
5.0             0.0011 ± 0.0009  0.0047 ± 0.0041  0.0057 ± 0.0030   
10.0            0.0012 ± 0.0010  0.0091 ± 0.0060  0.0122 ± 0.0043   
20.0            0.0019 ± 0.0014  0.0149 ± 0.0087  0.0269 ± 0.0073   
30.0            0.0029 ± 0.0019  0.0280 ± 0.0169  0.0455 ± 0.0093   
50.0            0.0053 ± 0.0037  0.0891 ± 0.0370  0.0948 ± 0.0120   
70.0            0.0071 ± 0.0056  0.2437 ± 0.0679  0.1680 ± 0.0144   
90.0            0.0176 ± 0.0075  0.8188 ± 0.2965  0.2310 ± 0.0150   

Graph Type                                                         \
Dataset                          citeseer_public                    
Model                                        GCN                    
#Layers                       4                2                4   
Drop Ratio (%)                                                      
1.0             0.0007 ± 0.0006  0.0006 ± 0.0004  0.0015 ± 0.0014   
5.0             0.0020 ± 0.0019  0.0012 ± 0.0008  0.0037 ± 0.0038   
10.0            0.0032 ± 0.0026  0.0022 ± 0.0016  0.0060 ± 0.0060   
20.0            0.0057 ± 0.0039  0.0038 ± 0.0021  0.0090 ± 0.0070   
30.0            0.0083 ± 0.0056  0.0056 ± 0.0035  0.0110 ± 0.0069   
50.0            0.0137 ± 0.0094  0.0107 ± 0.0065  0.0350 ± 0.0219   
70.0            0.0164 ± 0.0136  0.0216 ± 0.0075  0.1256 ± 0.0481   
90.0            0.5666 ± 2.2254  0.0460 ± 0.0081  0.4587 ± 0.1044   

Graph Type                                           Heterophilic  \
Dataset                                                     texas   
Model                       GAT                               GCN   
#Layers                       2                4                2   
Drop Ratio (%)                                                      
1.0             0.0028 ± 0.0023  0.0034 ± 0.0026  0.0173 ± 0.0259   
5.0             0.0133 ± 0.0052  0.0144 ± 0.0068  0.0256 ± 0.0245   
10.0            0.0281 ± 0.0073  0.0297 ± 0.0099  0.0334 ± 0.0278   
20.0            0.0606 ± 0.0114  0.0675 ± 0.0159  0.0417 ± 0.0364   
30.0            0.0966 ± 0.0130  0.1091 ± 0.0170  0.0500 ± 0.0351   
50.0            0.1853 ± 0.0150  0.2187 ± 0.0209  0.0850 ± 0.0581   
70.0            0.2942 ± 0.0149  0.3580 ± 0.0250  0.1589 ± 0.0706   
90.0            0.4341 ± 0.0128  0.5457 ± 0.0443  0.3814 ± 0.0602   

Graph Type                                                           \
Dataset                                                               
Model                                        GAT                      
#Layers                       4                2                  4   
Drop Ratio (%)                                                        
1.0             0.0069 ± 0.0094  0.0318 ± 0.0321    0.0296 ± 0.0273   
5.0             0.0271 ± 0.0202  0.1565 ± 0.1451    0.0732 ± 0.0613   
10.0            0.0393 ± 0.0341  0.3291 ± 0.2778    0.1383 ± 0.0969   
20.0            0.0631 ± 0.0504  0.5247 ± 0.3923    0.3874 ± 0.9694   
30.0            0.1001 ± 0.0635  0.7620 ± 0.6785    1.1412 ± 3.1941   
50.0            0.1566 ± 0.1061  0.8485 ± 0.4224   6.8657 ± 19.5882   
70.0            0.2491 ± 0.1253  0.9349 ± 0.3794                NaN   
90.0            0.1200 ± 0.0906  0.8721 ± 0.2932  23.4418 ± 39.3950   

Graph Type                                                         \
Dataset                 cornell                                     
Model                       GCN                               GAT   
#Layers                       2                4                2   
Drop Ratio (%)                                                      
1.0             0.0058 ± 0.0054  0.00

In [13]:
table_mape

Graph Type                Homophilic                       \
Dataset                  cora_public                        
Model                            GCN                        
#Layers                            2                    4   
Drop Ratio (%)                                              
1.0              199.1963 ± 535.6479  406.3717 ± 696.8121   
5.0             484.3248 ± 2363.7112  370.6094 ± 506.1477   
10.0             140.5256 ± 543.6766  271.4030 ± 484.8840   
20.0             191.1358 ± 468.1051  145.6388 ± 179.5958   
30.0            257.0281 ± 1074.6961    91.7429 ± 99.7362   
50.0             290.9328 ± 811.6082    79.7896 ± 22.0232   
70.0             122.2886 ± 177.2829     89.8458 ± 9.9204   
90.0               73.5859 ± 30.4604     95.4704 ± 2.5159   

Graph Type                                                  \
Dataset                                                      
Model                           GAT                          
#Layers                           2                      4   
Drop Ratio (%)                                               
1.0             225.6820 ± 928.6889    185.3690 ± 315.4137   
5.0               75.3701 ± 47.5184  1429.4159 ± 9135.1855   
10.0              70.7653 ± 31.4239    191.0706 ± 991.7390   
20.0              67.7009 ± 18.6599     72.5068 ± 217.2111   
30.0              65.4820 ± 12.1768      28.1198 ± 25.0595   
50.0               61.5370 ± 8.4232      15.9446 ± 11.7072   
70.0               57.1088 ± 5.5572        8.6027 ± 6.9418   
90.0               46.0412 ± 3.5192      28.7567 ± 24.3300   

Graph Type                                                                   \
Dataset             citeseer_public                                           
Model                           GCN                                     GAT   
#Layers                           2                    4                  2   
Drop Ratio (%)                                                                
1.0             115.8615 ± 148.7880  130.8737 ± 140.1379  82.1899 ± 44.0804   
5.0             196.0261 ± 577.3506  179.5552 ± 360.7461  92.1662 ± 40.2921   
10.0             95.0748 ± 137.3950  392.5160 ± 946.6271  87.1330 ± 20.9054   
20.0            148.0274 ± 258.1685  246.3180 ± 480.0688   83.7978 ± 9.2741   
30.0            106.0992 ± 202.3350  160.5323 ± 333.5508   82.7299 ± 6.3805   
50.0             83.7142 ± 238.8674    67.2995 ± 52.2963   82.2025 ± 4.2233   
70.0             81.9425 ± 130.5722    85.2427 ± 17.9479   80.9278 ± 3.6572   
90.0              66.4329 ± 10.0634     95.0904 ± 4.4708   78.2328 ± 1.8685   

Graph Type                                   Heterophilic  \
Dataset                                             texas   
Model                                                 GCN   
#Layers                          4                      2   
Drop Ratio (%)                                              
1.0              94.8345 ± 70.8957  1253.6573 ± 5157.1337   
5.0             100.9868 ± 66.3939   576.6617 ± 3105.5908   
10.0             87.2565 ± 36.3830  1037.9961 ± 4202.3550   
20.0             81.2137 ± 11.2173    328.4327 ± 933.0911   
30.0              80.3103 ± 8.8497    189.4972 ± 263.9378   
50.0              81.4559 ± 5.2121    369.8977 ± 732.8126   
70.0              81.2100 ± 3.2901    349.0767 ± 656.1638   
90.0              79.9629 ± 1.9104     116.0416 ± 18.2930   

Graph Type                                                      \
Dataset                                                          
Model                                                      GAT   
#Layers                            4                         2   
Drop Ratio (%)                                                   
1.0              192.3641 ± 317.0648      610.1125 ± 1522.1715   
5.0              308.7278 ± 469.9723       368.9435 ± 542.2813   
10.0            503.9676 ± 1097.2040     1087.9642 ± 5286.5876   
20.0             441.8645 ± 919.5732  37863.8353 

In [14]:
table_corr

Graph Type      Homophilic                                                  \
Dataset        cora_public                         citeseer_public           
Model                  GCN             GAT                     GCN           
#Layers                  2       4       2       4               2       4   
Drop Ratio (%)                                                               
1.0                 0.8678  0.6957  0.6874  0.7676          0.7868  0.6400   
5.0                 0.7844  0.7307  0.7736  0.8738          0.8822  0.7145   
10.0                0.9019  0.7423  0.7832  0.9405          0.7930  0.7397   
20.0                0.8663  0.7327  0.6497  0.9211          0.7368  0.6873   
30.0                0.8067  0.6255  0.6665  0.9145          0.7450  0.6641   
50.0                0.6873  0.5325  0.8013  0.8937          0.6340  0.4699   
70.0                0.6541  0.5045  0.6524  0.7618          0.7758  0.5492   
90.0                0.3998  0.4858  0.7573  0.4809          0.5903  0.1213   

Graph Type                     Heterophilic                                  \
Dataset                               texas                         cornell   
Model              GAT                  GCN             GAT             GCN   
#Layers              2       4            2       4       2       4       2   
Drop Ratio (%)                                                                
1.0             0.4332  0.5970       0.4873  0.6885  0.7503  0.3822  0.7499   
5.0             0.6133  0.7363       0.7329  0.3129  0.3356  0.5502  0.8258   
10.0            0.5276  0.6606       0.6176  0.3209  0.3950  0.4088  0.8334   
20.0            0.5304  0.4843       0.6585  0.4330  0.2822  0.3566  0.8171   
30.0            0.3653  0.5009       0.6795  0.3530  0.1533  0.3571  0.8746   
50.0            0.4002  0.5306       0.7875  0.2321  0.2705  0.1035  0.8072   
70.0            0.5269  0.5823       0.6427  0.0894  0.1942     NaN  0.6351   
90.0            0.6506  0.6649       0.6414  0.2525  0.1852  0.0769  0.4901   

Graph Type                                
Dataset                                   
Model                       GAT           
#Layers              4        2        4  
Drop Ratio (%)                            
1.0             0.5660   0.4729  -0.0237  
5.0             0.5322   0.2729   0.2640  
10.0            0.5745   0.4565   0.1779  
20.0            0.2642   0.4356   0.2938  
30.0            0.2146   0.3352   0.0444  
50.0            0.4368   0.1964   0.2081  
70.0            0.4936   0.0955  -0.0667  
90.0            0.1422  -0.0838  -0.1109

## Exp 3

In [15]:
df_exp3

,exp,experiment_name,result_dir,candidate_table,run_repeat_idx,setting_key,candidate_idx,candidate_edges,num_edges,calced_inf,...,cfg_removal_candidate_sampler,cfg_removal_neighbor_dist,cfg_num_of_clusters,cfg_edges_per_cluster,cfg_cluster_ratio_percent,cfg_intra_cluster_dist,cfg_inter_cluster_dist,cfg_influence_mode,cfg_num_group_elem,cfg_seed
0,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GAT/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""citeseer_public"", ""influence_mode...",0,1-486;1-2933;12-557;12-677;12-794;12-2474;12-2...,683,0.000083,...,group_neighbor,1,3,-1,15,1.0,1.0,calculate_influence,683,0
1,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GAT/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""citeseer_public"", ""influence_mode...",0,464-1758;1012-2785;453-887;1117-2596;2264-2638...,683,-0.001514,...,group_neighbor,4,3,-1,15,1.0,1.0,calculate_influence,683,0
2,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GAT/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""citeseer_public"", ""influence_mode...",1,1-1097;8-1033;18-582;18-778;27-229;27-2023;28-...,683,0.004829,...,group_neighbor,1,3,-1,15,1.0,1.0,calculate_influence,683,0
3,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GAT/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""citeseer_public"", ""influence_mode...",1,303-719;1130-2110;1251-2483;2775-2925;627-2681...,683,0.011816,...,group_neighbor,4,3,-1,15,1.0,1.0,calculate_influence,683,0
4,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GAT/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""citeseer_public"", ""influence_mode...",2,1-158;1-486;1-2919;10-2622;12-113;12-557;12-67...,683,0.012618,...,group_neighbor,1,3,-1,15,1.0,1.0,calculate_influence,683,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GCN/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""texas"", ""influence_mode"": ""calcul...",49,22-57;37-56;56-155;56-77;56-150;56-104;56-62;5...,44,-0.019082,...,group_neighbor,1,3,-1,15,1.0,1.0,calculate_influence,44,0
2196,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GCN/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""texas"", ""influence_mode"": ""calcul...",49,31-84;79-99;85-111;66-80;6-171;84-153;21-36;56...,44,-0.060619,...,group_neighbor,2,3,-1,15,1.0,1.0,calculate_influence,44,0
2197,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GCN/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""texas"", ""influence_mode"": ""calcul...",49,56-147;47-180;64-82;22-74;15-165;90-90;5-56;15...,44,-0.068233,...,group_neighbor,3,3,-1,15,1.0,1.0,calculate_influence,44,0
2198,exp3,non_neighbor_edges,results/GNH/mean_validation_loss/GCN/linear_Fa...,candidate_results.csv,0,"{""dataset"": ""texas"", ""influence_mode"": ""calcul...",49,85-111;58-63;0-121;5-56;80-176;56-82;24-40;56-...,44,-0.111834,...,group_neighbor,4,3,-1,15,1.0,1.0,calculate_influence,44,0


In [16]:
df3 = df_exp3.drop(columns=["candidate_table", "config", "setting_key", "experiment_name", "result_dir", "candidate_table", "setting_key", "candidate_idx", "candidate_edges"], inplace=False)
df3["inf_mae"] = (df3["calced_inf"] - df3["actual_inf"]).abs()
df3["inf_mape"] = (df3["calced_inf"] - df3["actual_inf"]).abs() / df3["actual_inf"].abs().replace(0, pd.NA) * 100
df3["inf_mape"] = pd.to_numeric(df3["inf_mape"], errors="coerce")

1
corr_group_keys = ["cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_removal_candidate_sampler", "cfg_removal_neighbor_dist", "run_repeat_idx"]
corr_by_run = (
    df3.groupby(corr_group_keys, dropna=False)
    .apply(lambda g: g["calced_inf"].corr(g["actual_inf"], method="pearson"))
    .rename("inf_corr")
    .reset_index()
)
df3 = df3.merge(corr_by_run, on=corr_group_keys, how="left")

/tmp/ipykernel_2738205/245177919.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df3.groupby(corr_group_keys, dropna=False)


In [17]:
df3['inf_corr']

0       0.676436
1       0.715242
2       0.676436
3       0.715242
4       0.676436
          ...   
2195    0.349899
2196    0.467638
2197    0.676521
2198    0.544556
2199    0.605926
Name: inf_corr, Length: 2200, dtype: float64

In [18]:
df3.columns

Index(['exp', 'run_repeat_idx', 'num_edges', 'calced_inf', 'actual_inf',
       'actual_source', 'cfg_dataset', 'cfg_model', 'cfg_num_layers',
       'cfg_ratio_group_elem', 'cfg_removal_candidate_sampler',
       'cfg_removal_neighbor_dist', 'cfg_num_of_clusters',
       'cfg_edges_per_cluster', 'cfg_cluster_ratio_percent',
       'cfg_intra_cluster_dist', 'cfg_inter_cluster_dist',
       'cfg_influence_mode', 'cfg_num_group_elem', 'cfg_seed', 'inf_mae',
       'inf_mape', 'inf_corr'],
      dtype='object')

In [19]:
df3['cfg_removal_neighbor_dist']

0       1
1       4
2       1
3       4
4       1
       ..
2195    1
2196    2
2197    3
2198    4
2199    5
Name: cfg_removal_neighbor_dist, Length: 2200, dtype: int64

In [20]:
df3_res = (
    df3.groupby(["cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_removal_candidate_sampler", "cfg_removal_neighbor_dist"])[["inf_mae", "inf_mape", "calced_inf", "actual_inf", "inf_corr"]]
    .agg(["mean", "std"])
)
# MultiIndex columns -> tuple index로 평탄화 (build_mean_std_table와 호환)
df3_res.columns = pd.Index(df3_res.columns.to_flat_index())

In [21]:
df3_res

(inf_mae, mean)  \
cfg_dataset     cfg_model cfg_num_layers cfg_removal_candidate_sampler cfg_removal_neighbor_dist                    
citeseer_public GAT       2              group_neighbor                1                                 0.046133   
                                                                       4                                 0.029377   
                GCN       2              group_neighbor                3                                 0.007933   
                                                                       4                                 0.002865   
                                                                       5                                 0.002073   
                                         group_non_neighbor            1                                 0.005847   
                          4              group_neighbor                1                                 0.008777   
                                                                       2                                 0.016360   
                                                                       3                                 0.039201   
                                                                       4                                 0.018133   
                                                                       5                                 0.012114   
                                         group_non_neighbor            1                                 0.005771   
                                                                       2                                 0.004885   
cora_public     GAT       2              group_neighbor                1                                 0.017600   
                                                                       3                                 0.036683   
                                                                       4                                 0.018122   
                                                                       5                                 0.012907   
                GCN       2              group_neighbor                3                                 0.004067   
                                                                       4                                 0.002645   
                                                                       5                                 0.001817   
                          4              group_neighbor                1                                 0.010880   
                                                                       3                                 0.006837   
                                                                       4                                 0.013269   
                                                                       5                                 0.009303   
cornell         GCN       2              group_neighbor                1                                 0.013336   
                                                                       2                                 0.019290   
                                                                       3                                 0.021185   
                                                                       4                                 0.021429   
                                                                       5                                 0.021912   
                          4              group_neighbor                1                                 0.048796   
                                                                       2                                 0.046331   
                                                                       3                                 0.032985   
                                                                       4                                 0.030477   
   

In [22]:
df3_res.to_csv("df3_res.csv")

In [23]:
import pandas as pd
import numpy as np


# =========================
# 설정
# =========================
HOMOPHILIC_DATASETS = ["cora_public", "citeseer_public"]
HETEROPHILIC_DATASETS = ["texas", "cornell"]

DATASET_ORDER = ["cora_public", "citeseer_public", "texas", "cornell"]
MODEL_ORDER = ["GCN", "GAT"]
LAYER_ORDER = [2, 4]
SAMPLER_ORDER = ["group_neighbor", "group_non_neighbor"]
DIST_ORDER = [1, 2, 3, 4, 5]

DATASET_NAME_MAP = {
    "cora_public": "Cora",
    "citeseer_public": "CiteSeer",
    "texas": "Texas",
    "cornell": "Cornell",
}

SAMPLER_NAME_MAP = {
    "group_neighbor": "Neighbor",
    "group_non_neighbor": "Non-neighbor",
}


# =========================
# metric 컬럼 이름 처리
# =========================
def metric_col(metric_name, stat):
    # csv를 거치면 컬럼명이 문자열 "('inf_mae', 'mean')" 꼴일 수 있음
    tuple_col = (metric_name, stat)
    str_col = f"('{metric_name}', '{stat}')"
    return tuple_col, str_col


def extract_metric_columns(df, metric_name):
    tuple_mean, str_mean = metric_col(metric_name, "mean")
    tuple_std, str_std = metric_col(metric_name, "std")

    if tuple_mean in df.columns and tuple_std in df.columns:
        mean_col, std_col = tuple_mean, tuple_std
    elif str_mean in df.columns and str_std in df.columns:
        mean_col, std_col = str_mean, str_std
    else:
        raise KeyError(
            f"Could not find metric columns for {metric_name}. "
            f"Expected either {tuple_mean}/{tuple_std} or {str_mean}/{str_std}"
        )
    return mean_col, std_col


# =========================
# pretty formatting
# =========================
def pretty_num(x):
    if x is None or pd.isna(x):
        return ""
    x = float(x)

    if x == 0:
        return "0"
    elif abs(x) < 0.01:
        return f"{x:.4f}"
    elif abs(x) < 1:
        return f"{x:.3f}"
    else:
        return f"{x:.2f}"


def format_mean_std_text(mean, std):
    if pd.isna(mean):
        return "--"
    mean_str = pretty_num(mean)
    if pd.isna(std):
        return mean_str
    std_str = pretty_num(std)
    return f"{mean_str} ± {std_str}"


def format_mean_std_html(mean, std):
    if pd.isna(mean):
        return "--"
    mean_str = pretty_num(mean)
    if pd.isna(std):
        return f'<span style="font-size:1em;font-weight:600;">{mean_str}</span>'
    std_str = pretty_num(std)
    return (
        f'<span style="font-size:1em;font-weight:600;">{mean_str}</span>'
        f'<span style="font-size:0.72em;color:#666;margin-left:0.2em;">± {std_str}</span>'
    )


def format_mean_std_latex(mean, std, bold=False):
    if pd.isna(mean):
        return "--"
    mean_str = pretty_num(mean)
    if pd.isna(std):
        return mean_str
    std_str = pretty_num(std)
    macro = r"\bmstd" if bold else r"\mstd"
    return f"{macro}{{{mean_str}}}{{{std_str}}}"


# =========================
# 공통 전처리
# =========================
def prepare_df3(
    df_res,
    metric_name,
    homophilic_datasets=None,
    heterophilic_datasets=None,
):
    df = df_res.copy().reset_index()

    mean_col, std_col = extract_metric_columns(df, metric_name)

    df["mean"] = df[mean_col]
    df["std"] = df[std_col]

    homophilic_datasets = homophilic_datasets or []
    heterophilic_datasets = heterophilic_datasets or []

    def dataset_group(ds):
        if ds in homophilic_datasets:
            return "Homophilic"
        elif ds in heterophilic_datasets:
            return "Heterophilic"
        return "Other"

    df["group"] = df["cfg_dataset"].map(dataset_group)
    return df


def ordered_columns(
    df,
    dataset_order=None,
    model_order=None,
    layer_order=None,
    sampler_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
):
    dataset_order = dataset_order or sorted(df["cfg_dataset"].unique())
    model_order = model_order or sorted(df["cfg_model"].unique())
    layer_order = layer_order or sorted(df["cfg_num_layers"].unique())
    sampler_order = sampler_order or sorted(df["cfg_removal_candidate_sampler"].unique())

    desired = []

    if homophilic_datasets:
        for ds in dataset_order:
            if ds not in homophilic_datasets:
                continue
            for model in model_order:
                for layer in layer_order:
                    for sampler in sampler_order:
                        desired.append(("Homophilic", ds, model, layer, sampler))

    if heterophilic_datasets:
        for ds in dataset_order:
            if ds not in heterophilic_datasets:
                continue
            for model in model_order:
                for layer in layer_order:
                    for sampler in sampler_order:
                        desired.append(("Heterophilic", ds, model, layer, sampler))

    remaining = [c for c in df.columns if c not in desired]
    return desired + remaining


# =========================
# 숫자 테이블
# row = neighbor_dist
# col = group / dataset / model / layers / sampler
# =========================
def build_numeric_tables_df3(
    df_res,
    metric_name,
    dist_order=None,
    dataset_order=None,
    model_order=None,
    layer_order=None,
    sampler_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
):
    df = prepare_df3(
        df_res,
        metric_name,
        homophilic_datasets=homophilic_datasets,
        heterophilic_datasets=heterophilic_datasets,
    )

    mean_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_removal_candidate_sampler"],
        values="mean",
    )

    std_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers", "cfg_removal_candidate_sampler"],
        values="std",
    )

    if dist_order is not None:
        mean_table = mean_table.reindex(dist_order)
        std_table = std_table.reindex(dist_order)

    col_order = ordered_columns(
        mean_table,
        dataset_order=dataset_order,
        model_order=model_order,
        layer_order=layer_order,
        sampler_order=sampler_order,
        homophilic_datasets=homophilic_datasets,
        heterophilic_datasets=heterophilic_datasets,
    )

    mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(col_order))
    std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(col_order))

    mean_table.index.name = "Neighbor Distance"
    std_table.index.name = "Neighbor Distance"

    mean_table.columns.names = ["Graph Type", "Dataset", "Model", "Layers", "Sampler"]
    std_table.columns.names = ["Graph Type", "Dataset", "Model", "Layers", "Sampler"]

    return mean_table, std_table


# =========================
# Jupyter용 text/html table
# =========================
def build_display_table_df3(
    df_res,
    metric_name="inf_mae",
    dist_order=None,
    dataset_order=None,
    model_order=None,
    layer_order=None,
    sampler_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    html=False,
):
    mean_table, std_table = build_numeric_tables_df3(
        df_res=df_res,
        metric_name=metric_name,
        dist_order=dist_order,
        dataset_order=dataset_order,
        model_order=model_order,
        layer_order=layer_order,
        sampler_order=sampler_order,
        homophilic_datasets=homophilic_datasets,
        heterophilic_datasets=heterophilic_datasets,
    )

    disp = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)

    for idx in mean_table.index:
        for col in mean_table.columns:
            m = mean_table.loc[idx, col]
            s = std_table.loc[idx, col]
            disp.loc[idx, col] = format_mean_std_html(m, s) if html else format_mean_std_text(m, s)

    disp.index.name = mean_table.index.name
    disp.columns.names = mean_table.columns.names
    return disp


# =========================
# Jupyter에서 보기 좋게
# =========================
def prettify_display_table(df):
    out = df.copy()

    new_cols = []
    for g, ds, model, layers, sampler in out.columns:
        ds = DATASET_NAME_MAP.get(ds, ds)
        sampler = SAMPLER_NAME_MAP.get(sampler, sampler)
        new_cols.append((g, ds, model, layers, sampler))

    out.columns = pd.MultiIndex.from_tuples(
        new_cols,
        names=out.columns.names,
    )

    out.index = [str(int(x)) if float(x).is_integer() else f"{x:g}" for x in out.index]
    out.index.name = "Neighbor Distance"
    return out


# =========================
# LaTeX용 table 생성
# highlight = None / "min" / "max"
# =========================
def build_latex_table_df3(
    df_res,
    metric_name="inf_mae",
    dist_order=None,
    dataset_order=None,
    model_order=None,
    layer_order=None,
    sampler_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    highlight=None,
):
    mean_table, std_table = build_numeric_tables_df3(
        df_res=df_res,
        metric_name=metric_name,
        dist_order=dist_order,
        dataset_order=dataset_order,
        model_order=model_order,
        layer_order=layer_order,
        sampler_order=sampler_order,
        homophilic_datasets=homophilic_datasets,
        heterophilic_datasets=heterophilic_datasets,
    )

    if highlight not in [None, "min", "max"]:
        raise ValueError("highlight must be one of None, 'min', 'max'")

    bold_mask = pd.DataFrame(False, index=mean_table.index, columns=mean_table.columns)

    if highlight == "min":
        for col in mean_table.columns:
            vals = mean_table[col].dropna()
            if len(vals) > 0:
                target = vals.min()
                bold_mask.loc[mean_table[col] == target, col] = True
    elif highlight == "max":
        for col in mean_table.columns:
            vals = mean_table[col].dropna()
            if len(vals) > 0:
                target = vals.max()
                bold_mask.loc[mean_table[col] == target, col] = True

    latex_df = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)

    for idx in mean_table.index:
        for col in mean_table.columns:
            latex_df.loc[idx, col] = format_mean_std_latex(
                mean_table.loc[idx, col],
                std_table.loc[idx, col],
                bold=bold_mask.loc[idx, col],
            )

    latex_df.index.name = "Neighbor Distance"
    latex_df.columns.names = mean_table.columns.names
    return latex_df


def build_display_table_df3_by_sampler(
    df_res,
    sampler_name,               # "group_neighbor" or "group_non_neighbor"
    metric_name="inf_mae",
    dist_order=None,
    dataset_order=None,
    model_order=None,
    layer_order=None,
    homophilic_datasets=None,
    heterophilic_datasets=None,
    html=False,
):
    df = df_res.copy()
    if "cfg_removal_candidate_sampler" not in df.columns and "cfg_removal_candidate_sampler" in df.index.names:
        df = df.reset_index()
        
    mean_col, std_col = extract_metric_columns(df, metric_name)
    df["mean"] = df[mean_col]
    df["std"] = df[std_col]

    # sampler 하나만 선택
    df = df[df["cfg_removal_candidate_sampler"] == sampler_name].copy()

    homophilic_datasets = homophilic_datasets or []
    heterophilic_datasets = heterophilic_datasets or []

    def dataset_group(ds):
        if ds in homophilic_datasets:
            return "Homophilic"
        elif ds in heterophilic_datasets:
            return "Heterophilic"
        return "Other"

    df["group"] = df["cfg_dataset"].map(dataset_group)

    # pivot: sampler 축 제거
    mean_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="mean",
    )

    std_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="std",
    )

    # row 순서
    if dist_order is not None:
        mean_table = mean_table.reindex(dist_order)
        std_table = std_table.reindex(dist_order)

    # column 순서
    dataset_order = dataset_order or sorted(df["cfg_dataset"].unique())
    model_order = model_order or sorted(df["cfg_model"].unique())
    layer_order = layer_order or sorted(df["cfg_num_layers"].unique())

    desired_cols = []

    if homophilic_datasets:
        for ds in dataset_order:
            if ds not in homophilic_datasets:
                continue
            for model in model_order:
                for layer in layer_order:
                    col = ("Homophilic", ds, model, layer)
                    desired_cols.append(col)

    if heterophilic_datasets:
        for ds in dataset_order:
            if ds not in heterophilic_datasets:
                continue
            for model in model_order:
                for layer in layer_order:
                    col = ("Heterophilic", ds, model, layer)
                    desired_cols.append(col)

    remaining_cols = [c for c in mean_table.columns if c not in desired_cols]
    desired_cols.extend(remaining_cols)

    mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))
    std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

    # display table 생성
    disp = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)

    for idx in mean_table.index:
        for col in mean_table.columns:
            m = mean_table.loc[idx, col]
            s = std_table.loc[idx, col]
            disp.loc[idx, col] = format_mean_std_html(m, s) if html else format_mean_std_text(m, s)

    disp.index.name = "Neighbor Distance"
    disp.columns.names = ["Graph Type", "Dataset", "Model", "Layers"]

    return disp

def prettify_sampler_split_table(df):
    out = df.copy()

    new_cols = []
    for g, ds, model, layers in out.columns:
        ds = DATASET_NAME_MAP.get(ds, ds)
        new_cols.append((g, ds, model, layers))

    out.columns = pd.MultiIndex.from_tuples(
        new_cols,
        names=["Graph Type", "Dataset", "Model", "Layers"]
    )

    out.index = [str(int(x)) if float(x).is_integer() else f"{x:g}" for x in out.index]
    out.index.name = "Neighbor Distance"

    return out

# =========================
# 사용 예시
# =========================

# 1) 화면용 plain text table
table_df3 = build_display_table_df3(
    df3_res,
    metric_name="inf_corr",   # or inf_mape / calced_inf / actual_inf / inf_corr
    dist_order=DIST_ORDER,
    dataset_order=DATASET_ORDER,
    model_order=MODEL_ORDER,
    layer_order=LAYER_ORDER,
    sampler_order=SAMPLER_ORDER,
    homophilic_datasets=HOMOPHILIC_DATASETS,
    heterophilic_datasets=HETEROPHILIC_DATASETS,
    html=False,
)
table_df3 = prettify_display_table(table_df3)
table_df3

Graph Type        Homophilic                                                \
Dataset                 Cora                         CiteSeer                
Model                    GCN                   GAT        GCN                
Layers                     2          4          2          2                
Sampler             Neighbor   Neighbor   Neighbor   Neighbor Non-neighbor   
Neighbor Distance                                                            
1                         --  0.527 ± 0  0.824 ± 0         --    0.719 ± 0   
2                         --         --         --         --           --   
3                  0.588 ± 0  0.236 ± 0  0.906 ± 0  0.723 ± 0           --   
4                  0.728 ± 0  0.026 ± 0  0.854 ± 0  0.605 ± 0           --   
5                  0.920 ± 0  0.543 ± 0  0.811 ± 0  0.893 ± 0           --   

Graph Type                                            Heterophilic             \
Dataset                                                      Texas              
Model                                             GAT          GCN              
Layers                      4                       2            2          4   
Sampler              Neighbor Non-neighbor   Neighbor     Neighbor   Neighbor   
Neighbor Distance                                                               
1                   0.552 ± 0    0.647 ± 0  0.676 ± 0    0.807 ± 0  0.350 ± 0   
2                   0.082 ± 0    0.499 ± 0         --    0.561 ± 0  0.468 ± 0   
3                  -0.202 ± 0           --         --    0.799 ± 0  0.677 ± 0   
4                   0.223 ± 0           --  0.715 ± 0    0.737 ± 0  0.545 ± 0   
5                   0.787 ± 0           --         --    0.753 ± 0  0.606 ± 0   

Graph Type                               
Dataset              Cornell             
Model                    GCN             
Layers                     2          4  
Sampler             Neighbor   Neighbor  
Neighbor Distance                        
1                  0.811 ± 0  0.180 ± 0  
2                  0.837 ± 0  0.228 ± 0  
3                  0.855 ± 0  0.325 ± 0  
4                  0.904 ± 0  0.424 ± 0  
5                  0.905 ± 0  0.584 ± 0

In [24]:
table_neighbor = prettify_sampler_split_table(
    build_display_table_df3_by_sampler(
        df3_res,
        sampler_name="group_neighbor",
        metric_name="inf_corr",
        dist_order=[1, 2, 3, 4, 5],
        dataset_order=["cora_public", "citeseer_public", "texas", "cornell"],
        model_order=["GCN", "GAT"],
        layer_order=[2, 4],
        homophilic_datasets=["cora_public", "citeseer_public"],
        heterophilic_datasets=["texas", "cornell"],
        html=False,
    )
)

table_non_neighbor = prettify_sampler_split_table(
    build_display_table_df3_by_sampler(
        df3_res,
        sampler_name="group_non_neighbor",
        metric_name="inf_corr",
        dist_order=[1, 2, 3, 4, 5],
        dataset_order=["cora_public", "citeseer_public", "texas", "cornell"],
        model_order=["GCN", "GAT"],
        layer_order=[2, 4],
        homophilic_datasets=["cora_public", "citeseer_public"],
        heterophilic_datasets=["texas", "cornell"],
        html=False,
    )
)

display(table_neighbor)
display(table_non_neighbor)

Graph Type        Homophilic                                               \
Dataset                 Cora                         CiteSeer               
Model                    GCN                   GAT        GCN               
Layers                     2          4          2          2           4   
Neighbor Distance                                                           
1                         --  0.527 ± 0  0.824 ± 0         --   0.552 ± 0   
2                         --         --         --         --   0.082 ± 0   
3                  0.588 ± 0  0.236 ± 0  0.906 ± 0  0.723 ± 0  -0.202 ± 0   
4                  0.728 ± 0  0.026 ± 0  0.854 ± 0  0.605 ± 0   0.223 ± 0   
5                  0.920 ± 0  0.543 ± 0  0.811 ± 0  0.893 ± 0   0.787 ± 0   

Graph Type                   Heterophilic                                   
Dataset                             Texas               Cornell             
Model                    GAT          GCN                   GCN             
Layers                     2            2          4          2          4  
Neighbor Distance                                                           
1                  0.676 ± 0    0.807 ± 0  0.350 ± 0  0.811 ± 0  0.180 ± 0  
2                         --    0.561 ± 0  0.468 ± 0  0.837 ± 0  0.228 ± 0  
3                         --    0.799 ± 0  0.677 ± 0  0.855 ± 0  0.325 ± 0  
4                  0.715 ± 0    0.737 ± 0  0.545 ± 0  0.904 ± 0  0.424 ± 0  
5                         --    0.753 ± 0  0.606 ± 0  0.905 ± 0  0.584 ± 0

Graph Type        Homophilic           
Dataset             CiteSeer           
Model                    GCN           
Layers                     2          4
Neighbor Distance                      
1                  0.719 ± 0  0.647 ± 0
2                         --  0.499 ± 0
3                         --         --
4                         --         --
5                         --         --

In [ ]:
metric_name = "inf_corr"
highlight = "max"  # inf_corr is better when larger


def make_exp3_sampler_latex(df_res, sampler_name, caption, label):
    df = df_res.reset_index().copy()
    mean_col, std_col = extract_metric_columns(df, metric_name)
    df["mean"] = df[mean_col]
    df["std"] = df[std_col]
    df = df[df["cfg_removal_candidate_sampler"] == sampler_name].copy()

    df["group"] = df["cfg_dataset"].map(
        lambda ds: "Homophilic" if ds in HOMOPHILIC_DATASETS else ("Heterophilic" if ds in HETEROPHILIC_DATASETS else "Other")
    )

    mean_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="mean",
    )
    std_table = df.pivot(
        index="cfg_removal_neighbor_dist",
        columns=["group", "cfg_dataset", "cfg_model", "cfg_num_layers"],
        values="std",
    )

    mean_table = mean_table.reindex(DIST_ORDER)
    std_table = std_table.reindex(DIST_ORDER)

    desired_cols = []
    for ds_group, ds_list in [("Homophilic", HOMOPHILIC_DATASETS), ("Heterophilic", HETEROPHILIC_DATASETS)]:
        for ds in DATASET_ORDER:
            if ds not in ds_list:
                continue
            for model in MODEL_ORDER:
                for layer in LAYER_ORDER:
                    col = (ds_group, ds, model, layer)
                    desired_cols.append(col)

    remaining_cols = [c for c in mean_table.columns if c not in desired_cols]
    desired_cols.extend(remaining_cols)
    mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))
    std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

    bold_mask = pd.DataFrame(False, index=mean_table.index, columns=mean_table.columns)
    for col in mean_table.columns:
        vals = mean_table[col].dropna()
        if len(vals) == 0:
            continue
        target = vals.max() if highlight == "max" else vals.min()
        bold_mask.loc[mean_table[col] == target, col] = True

    latex_df = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)
    for idx in mean_table.index:
        for col in mean_table.columns:
            latex_df.loc[idx, col] = format_mean_std_latex(
                mean_table.loc[idx, col],
                std_table.loc[idx, col],
                bold=bold_mask.loc[idx, col],
            )

    pretty_cols = []
    for g, ds, model, layers in latex_df.columns:
        pretty_cols.append((g, DATASET_NAME_MAP.get(ds, ds), model, layers))
    latex_df.columns = pd.MultiIndex.from_tuples(
        pretty_cols,
        names=["Graph Type", "Dataset", "Model", "Layers"],
    )

    latex_df_str = latex_df.copy()
    latex_df_str.index = [str(int(x)) if float(x).is_integer() else f"{x:g}" for x in latex_df.index]
    latex_df_str.index.name = "Neighbor Distance"

    dataset_block_sizes = []
    prev_key = None
    for g, ds, _, _ in latex_df_str.columns:
        key = (g, ds)
        if key != prev_key:
            dataset_block_sizes.append(1)
            prev_key = key
        else:
            dataset_block_sizes[-1] += 1

    column_format = "l|" + "|".join("c" * sz for sz in dataset_block_sizes) + "|"

    latex_str = latex_df_str.to_latex(
        escape=False,
        multicolumn=True,
        multirow=True,
        column_format=column_format,
        caption=caption,
        label=label,
    )
    return latex_df_str, latex_str


latex_df3_neighbor, latex_str_exp3_neighbor = make_exp3_sampler_latex(
    df3_res,
    sampler_name="group_neighbor",
    caption="Exp3 influence correlation for neighbor candidate sampling.",
    label="tab:exp3_inf_corr_neighbor",
)

latex_df3_non_neighbor, latex_str_exp3_non_neighbor = make_exp3_sampler_latex(
    df3_res,
    sampler_name="group_non_neighbor",
    caption="Exp3 influence correlation for non-neighbor candidate sampling.",
    label="tab:exp3_inf_corr_non_neighbor",
)

print(latex_str_exp3_neighbor)
print()
print(latex_str_exp3_non_neighbor)


## Exp 4

In [25]:
df4 = df_exp4.drop(columns=["candidate_table", "config", "setting_key", "experiment_name", "result_dir", "candidate_table", "setting_key", "candidate_idx", "candidate_edges"], inplace=False)
df4["inf_mae"] = (df4["calced_inf"] - df4["actual_inf"]).abs()
df4["inf_mape"] = (df4["calced_inf"] - df4["actual_inf"]).abs() / df4["actual_inf"].abs().replace(0, pd.NA) * 100
df4["inf_mape"] = pd.to_numeric(df4["inf_mape"], errors="coerce")

1
corr_group_keys = ["cfg_dataset", "cfg_model", "cfg_num_layers", 'cfg_intra_cluster_dist', 'cfg_inter_cluster_dist',]
corr_by_run = (
    df4.groupby(corr_group_keys, dropna=False)
    .apply(lambda g: g["calced_inf"].corr(g["actual_inf"], method="pearson"))
    .rename("inf_corr")
    .reset_index()
)
df4 = df4.merge(corr_by_run, on=corr_group_keys, how="left")

/tmp/ipykernel_2738205/3481102794.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df4.groupby(corr_group_keys, dropna=False)


In [26]:
df4.columns

Index(['exp', 'run_repeat_idx', 'num_edges', 'calced_inf', 'actual_inf',
       'actual_source', 'cfg_dataset', 'cfg_model', 'cfg_num_layers',
       'cfg_ratio_group_elem', 'cfg_removal_candidate_sampler',
       'cfg_removal_neighbor_dist', 'cfg_num_of_clusters',
       'cfg_edges_per_cluster', 'cfg_cluster_ratio_percent',
       'cfg_intra_cluster_dist', 'cfg_inter_cluster_dist',
       'cfg_influence_mode', 'cfg_num_group_elem', 'cfg_seed', 'inf_mae',
       'inf_mape', 'inf_corr'],
      dtype='object')

In [27]:
df4_res = (
    df4.groupby(["cfg_dataset", "cfg_model", "cfg_num_layers", 'cfg_intra_cluster_dist', 'cfg_inter_cluster_dist',])[["inf_mae", "inf_mape", "calced_inf", "actual_inf", "inf_corr"]]
    .agg(["mean", "std"])
)
# MultiIndex columns -> tuple index로 평탄화 (build_mean_std_table와 호환)
df4_res.columns = pd.Index(df4_res.columns.to_flat_index())

In [28]:
pd.set_option('display.max_rows', None)

In [29]:
df4_res

(inf_mae, mean)  \
cfg_dataset     cfg_model cfg_num_layers cfg_intra_cluster_dist cfg_inter_cluster_dist                    
citeseer_public GAT       2              1.0                    2.0                            0.046378   
                GCN       2              1.0                    1.0                            0.003493   
                                                                2.0                            0.004702   
                                                                3.0                            0.005377   
cora_public     GAT       2              1.0                    2.0                            0.024262   
                GCN       4              1.0                    1.0                            0.012166   
                                                                2.0                            0.015484   
cornell         GAT       2              1.0                    1.0                            0.320510   
                                         2.0                    1.0                            0.198614   
                                                                2.0                            0.257985   
                                         3.0                    1.0                            0.182736   
                                                                2.0                            0.246373   
                          4              1.0                    1.0                            1.118258   
                                                                2.0                            0.581816   
                                         2.0                    1.0                            0.559706   
                                                                2.0                            0.809747   
                                         3.0                    1.0                            0.827878   
                                                                2.0                           31.096867   
                GCN       2              1.0                    1.0                            0.033819   
                                         2.0                    1.0                            0.029998   
                                                                2.0                            0.061825   
                                         3.0                    1.0                            0.026035   
                                                                2.0                            0.070660   
                          4              1.0                    1.0                            0.022613   
                                         2.0                    1.0                            0.031669   
                                                                2.0                            0.030241   
                                         3.0                    1.0                            0.027512   
                                                                2.0                            0.034845   
texas           GAT       2              1.0                    1.0                            0.441287   
                                         2.0                    1.0                            0.400967   
                                         3.0                    1.0                            0.523020   
                          4              1.0                    1.0                            0.460378   
                                         2.0                    1.0                            0.284832   
                                         3.0                    1.0                            0.485917   
                GCN       2              1.0                    1.0                            0.043669   
                                                                2.0                            0.039713   
                      

In [30]:
metric_name = 'inf_corr'
highlight = 'max'  # inf_corr는 클수록 좋음

df4_latex_src = df4_res.reset_index().copy()
mean_col, std_col = extract_metric_columns(df4_latex_src, metric_name)
df4_latex_src['mean'] = df4_latex_src[mean_col]
df4_latex_src['std'] = df4_latex_src[std_col]

df4_latex_src['group'] = df4_latex_src['cfg_dataset'].map(
    lambda ds: 'Homophilic' if ds in HOMOPHILIC_DATASETS else ('Heterophilic' if ds in HETEROPHILIC_DATASETS else 'Other')
)

mean_table = df4_latex_src.pivot(
    index=['cfg_intra_cluster_dist', 'cfg_inter_cluster_dist'],
    columns=['group', 'cfg_dataset', 'cfg_model', 'cfg_num_layers'],
    values='mean',
)
std_table = df4_latex_src.pivot(
    index=['cfg_intra_cluster_dist', 'cfg_inter_cluster_dist'],
    columns=['group', 'cfg_dataset', 'cfg_model', 'cfg_num_layers'],
    values='std',
)

cluster_dist_order = [1, 2, 3]
row_order = pd.MultiIndex.from_product(
    [cluster_dist_order, cluster_dist_order],
    names=['Intra Dist', 'Inter Dist'],
)
mean_table = mean_table.reindex(row_order)
std_table = std_table.reindex(row_order)

desired_cols = []
for ds_group, ds_list in [('Homophilic', HOMOPHILIC_DATASETS), ('Heterophilic', HETEROPHILIC_DATASETS)]:
    for ds in DATASET_ORDER:
        if ds not in ds_list:
            continue
        for model in MODEL_ORDER:
            for layer in LAYER_ORDER:
                col = (ds_group, ds, model, layer)
                if col in mean_table.columns:
                    desired_cols.append(col)

remaining_cols = [c for c in mean_table.columns if c not in desired_cols]
desired_cols.extend(remaining_cols)
mean_table = mean_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))
std_table = std_table.reindex(columns=pd.MultiIndex.from_tuples(desired_cols))

bold_mask = pd.DataFrame(False, index=mean_table.index, columns=mean_table.columns)
for col in mean_table.columns:
    vals = mean_table[col].dropna()
    if len(vals) == 0:
        continue
    target = vals.max() if highlight == 'max' else vals.min()
    bold_mask.loc[mean_table[col] == target, col] = True

latex_df4 = pd.DataFrame(index=mean_table.index, columns=mean_table.columns, dtype=object)
for idx in mean_table.index:
    for col in mean_table.columns:
        latex_df4.loc[idx, col] = format_mean_std_latex(
            mean_table.loc[idx, col],
            std_table.loc[idx, col],
            bold=bold_mask.loc[idx, col],
        )

pretty_cols = []
for g, ds, model, layers in latex_df4.columns:
    pretty_cols.append((g, DATASET_NAME_MAP.get(ds, ds), model, layers))
latex_df4.columns = pd.MultiIndex.from_tuples(
    pretty_cols,
    names=['Graph Type', 'Dataset', 'Model', 'Layers'],
)

latex_df4_str = latex_df4.copy()
latex_df4_str.index = pd.MultiIndex.from_tuples(
    [(str(int(i)), str(int(j))) for i, j in latex_df4.index],
    names=['Intra Dist', 'Inter Dist'],
)

dataset_block_sizes = []
prev_key = None
for g, ds, _, _ in latex_df4_str.columns:
    key = (g, ds)
    if key != prev_key:
        dataset_block_sizes.append(1)
        prev_key = key
    else:
        dataset_block_sizes[-1] += 1

column_format = 'll|' + '|'.join('c' * sz for sz in dataset_block_sizes) + '|'

latex_str_exp4 = latex_df4_str.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True,
    column_format=column_format,
    caption='Exp4 influence correlation by intra/inter cluster distances.',
    label='tab:exp4_inf_corr',
)

print(latex_str_exp4)

\begin{table}
\caption{Exp4 influence correlation by intra/inter cluster distances.}
\label{tab:exp4_inf_corr}
\begin{tabular}{ll|cc|cc|cccc|cccc|}
\toprule
 & Graph Type & \multicolumn{4}{r}{Homophilic} & \multicolumn{8}{r}{Heterophilic} \\
 & Dataset & \multicolumn{2}{r}{Cora} & \multicolumn{2}{r}{CiteSeer} & \multicolumn{4}{r}{Texas} & \multicolumn{4}{r}{Cornell} \\
 & Model & GCN & GAT & GCN & GAT & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} & \multicolumn{2}{r}{GCN} & \multicolumn{2}{r}{GAT} \\
 & Layers & 4 & 2 & 2 & 2 & 2 & 4 & 2 & 4 & 2 & 4 & 2 & 4 \\
Intra Dist & Inter Dist &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{1} & 1 & \bmstd{0.680}{0} & -- & \bmstd{0.736}{0} & -- & \mstd{0.547}{0} & \mstd{0.462}{0} & \mstd{-0.163}{0} & \bmstd{0.524}{0} & \mstd{0.553}{0} & \bmstd{0.562}{0} & \mstd{-0.224}{0} & \mstd{-0.135}{0} \\
 & 2 & \mstd{0.607}{0} & \bmstd{0.928}{0} & \mstd{0.672}{0} & \bmstd{0.711}{0} & \bmstd{0.903}{0} & \mstd{0.494}{0} & -- & -- & -

## Exp 5